# Lesson 3.6 — History Representation：History 到底怎么喂给 Neural Network？

3.5 已经回答了**为什么**需要 history：observation 丢失信息时，同一个 $o_t$ 会对应不同的正确 action，单帧 policy 只能在信息不足的条件下输出条件均值。

本课追问下一个问题：**history 到底存在哪里？模型又是怎么把它取回来的？**

> 不同架构的差别不在于"要不要 history"，而在于 **history 是被物理保留、还是被压缩**，以及**取回它的成本结构**。

对应 `docs/roadmap_v3.md` 的 3.6（单帧策略与历史策略），是 3.5 §5.4 那一行对比的展开。

学完本课后应该能：

1. 说出 Frame Stacking / RNN / LSTM / Transformer 四种表示法各自的 **history 存储位置**；
2. 解释 RNN 与 LSTM 同属"压缩派"，但 LSTM 多解决了什么；
3. 解释 attention 的 $QK^\top$ 为什么会得到 $[T,T]$ 矩阵，以及**第 $i$ 行**代表什么、它还需要哪一步才成为"关注程度"；
4. 说出 KV cache 是什么，以及它为什么让 Transformer 的 history 是"保留"而不是"压缩"；
5. 在两个轴上比较四种方法：**history 能不能被无损保留**、**时间维能不能并行**。

## 1. 同一个问题，四种答案

同样是把 history 喂给 network，四种方法给出了四种不同答案：

| 方法 | 一句话核心 |
|---|---|
| Frame Stacking | 把过去直接摆在桌面上 |
| RNN | 把过去压缩成一个 hidden state |
| LSTM | 给压缩记忆增加读写/遗忘门 |
| Transformer | 保留过去 token，需要时直接查 |

这四种可以先按一句话分成两派：

- **保留派：Frame Stacking / Transformer** —— history 以原始输入或 token 的形式物理存在；
- **压缩派：RNN / LSTM** —— 全部历史被塞进一个固定大小的 state。

"保留 vs 压缩"是本课后面所有对比的主轴。

## 2. 四种方法放在一起

| 方法 | History 存在哪里？ | History 长度 | 获取过去信息的方法 | 核心限制 |
|---|---|---|---|---|
| Frame Stacking | 原始输入 buffer | 固定 $k$ | concat | 超过窗口直接消失 |
| RNN | $h_t$ | 理论无限 | recursive compression | 长期依赖难学 |
| LSTM | $h_t, c_t$ | 理论无限 | gated recurrent memory | 仍是压缩瓶颈 |
| Transformer | historical tokens / KV | context window | attention retrieval | context 有限，标准 attention 成本随长度快速增长 |

表里的每一行下面单独展开。

## 3. 压缩派：RNN 与 LSTM

### 3.1 RNN：把整个过去压进一个 hidden state

$$
h_t = f(W h_{t-1} + U o_t + b)
$$

更准确的说法是：

$$
\boxed{\text{RNN compresses the whole past into recurrent state}}
$$

这带来两个后果：

1. **容量瓶颈**：无论过去多长，都只能通过固定维度的 $h_t$ 传递，信息多的任务必然丢东西；
2. **梯度路径太长**：从 $o_1$ 到 $h_T$ 要连续穿过 $T-1$ 次同一个矩阵乘法和同一个非线性，梯度会消失或爆炸。

第二点才是 LSTM 真正要解决的问题。

### 3.2 LSTM 的关键设计：给 memory 和梯度留一条近似线性的通路

这就是 LSTM 最重要的设计思想之一：

```text
create a relatively linear path for memory and gradient flow
```

**为什么"加法"就能给出线性通路？** 看 cell state 的更新式：

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t
$$

沿这条对角路径对 $c_{t-1}$ 求导：

$$
\frac{\partial c_t}{\partial c_{t-1}} = \operatorname{diag}(f_t) + (\text{经过 gate 的项})
$$

当 $f_t \approx 1$ 时，这条路径上的梯度**既不乘权重矩阵、也不经过 $\tanh$ 的饱和区**，近似恒等传递。对比 RNN：

$$
\frac{\partial h_t}{\partial h_{t-1}} = \operatorname{diag}\!\left(1-h_t^2\right) W
$$

它每一步都要乘一遍完整的 $W$，并且被 $1-h_t^2 \le 1$ 反复衰减。

所以 LSTM 比 RNN 更能记住长期信息，原因**不是它记得更多，而是梯度更容易流回去**——它没有解除压缩，只是让压缩过程可训练。

（精确地说，$f_t, i_t, \tilde c_t, g_t^{out}$ 都依赖 $h_{t-1}$，而 $h_{t-1}$ 又依赖 $c_{t-1}$，所以还存在其它项；上面写的是主导的对角路径。）

### 3.3 LSTM 的三个 gate

> **记号约定**：本仓库用 $o_t$ 表示时刻 $t$ 的 **observation**（与 3.1–3.5 一致）。原始文献里 LSTM 的输入写 $x_t$、而**输出门写 $o_t$**，两者一撞就会混淆。本课统一为：输入用 $o_t$，输出门用 $g_t^{out}$。

**Forget gate** —— 决定旧信息删多少：

$$
f_t = \sigma(W_f[o_t; h_{t-1}] + b_f)
$$

**Input gate** —— 决定新信息写多少，同时生成 candidate memory：

$$
i_t = \sigma(W_i[o_t; h_{t-1}] + b_i),
\qquad
\tilde c_t = \tanh(W_c[o_t; h_{t-1}] + b_c)
$$

**更新 cell state** —— 注意这一步是加法（见 3.2）：

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t
$$

**Output gate** —— 决定 cell state 里哪些信息现在暴露成 hidden state：

$$
g_t^{out} = \sigma(W_o[o_t; h_{t-1}] + b_o),
\qquad
h_t = g_t^{out} \odot \tanh(c_t)
$$

其中 $[\,\cdot\,;\,\cdot\,]$ 表示向量拼接，$\odot$ 为逐元素乘。

三个 gate 的分工：

| Gate | 管什么 | 类比 |
|---|---|---|
| $f_t$ | 旧记忆保留多少 | 擦除 |
| $i_t$ | 新信息写入多少 | 写入 |
| $g_t^{out}$ | 记忆暴露多少给外部 | 读取 |

关键区分：$c_t$ 是**内部长期记忆**（不直接对外），$h_t$ 是**对外输出**（也是下一时刻的输入之一）。这就是对比表里 LSTM 的 history 位置写成 $h_t, c_t$ 两个量的原因。

## 4. 检索派：Transformer 的 Attention

经典公式：

$$
\boxed{ Attention(Q,K,V) = softmax \left( \frac{QK^\top}{\sqrt{d_k}} \right)V }
$$

其中 $QK^\top$ 会产生一个 $[T,T]$ 的矩阵。假设 $T=10$，那么每个 sequence 都会有一个 $10\times10$ 的矩阵。

这里必须分开两个不同的对象，否则会读错：

| 对象 | 公式 | 第 $i$ 行的含义 | 性质 |
|---|---|---|---|
| **score matrix** | $S = QK^\top/\sqrt{d_k}$ | 第 $i$ 个 token 给每个 token 打的**原始分数** | 可正可负；行和没有约束；**行之间不可比较** |
| **attention weights** | $A = \operatorname{softmax}(S)$ | 第 $i$ 个 token 对其他 token 的**关注程度** | 非负；**每行和恒为 1** |

所以"第 $i$ 行表示第 $i$ 个 token 对其他 token 的关注程度"这句话描述的是 $A$，不是 $QK^\top$——**中间少了 softmax 这一步**。

两个直接推论：

1. **$A$ 的每一行是一个概率分布**，$O = AV$ 的每一行是 value 行的凸组合。也就是说 attention 只做**混合与搬运**，不做变换；真正的变换发生在它后面的 FFN。
2. **$A$ 是 $[T,T]$，所以成本约 $O(T^2)$**。这正是对比表里 Transformer "context 有限、成本随长度快速增长"的来源。

更细的机制（$Q/K/V$ 三种角色、$1/\sqrt{d_k}$ 为什么是温度、mask、multi-head、每个 head 的表达力上限）见 `notes/concepts.md` 的 "Transformer and Attention Fundamentals"，以及可运行的验证 fixture：

```bash
python scripts/attention_walkthrough.py   # 23 条断言，纯标准库，无需 numpy
```

本课不重复那部分，只关心它和 RNN/LSTM 在"历史放在哪里"上的根本区别：

$$
\boxed{\text{Transformer retains distributed token-level history}}
$$

RNN 的 $h_t$ 是一个**汇聚点**：所有过去必须经过它。Transformer 的历史是**分布式**的：每个过去 token 都还活着，需要时由 attention 直接检索。这就是"保留"与"压缩"的差别。

### 4.1 KV cache：Transformer 的 history 到底存在哪里

对比表里 Transformer 的 history 位置写的是 "historical token representations / KV cache"。前半句是训练时的情形，后半句是推理时的实现。

自回归解码时，每生成一个新 token 都要对**全部**历史做一次 attention。如果每步都重算所有历史的 $K, V$，总代价是 $O(T^2)$。**KV cache** 的做法是缓存每个已生成 token 的 $K, V$：

- 新 token 只计算自己的 $q, k, v$；
- 然后对**缓存的** $K, V$ 做一次 attention；
- 每步新增计算量 $O(T)$，但显存里的 cache 随 $T$ **线性增长**。

所以 KV cache 就是"history 被物理保留"这句话的具体形态：历史没有被压进任何一个向量，而是以 key/value 的形式**躺着**，等 query 来查。这也是长上下文贵的原因——贵在显存，而不只在算力。

| | 推理时每步显存 | history 信息 |
|---|---|---|
| RNN / LSTM | 固定（只要 $h_t$ / $h_t, c_t$） | 已被压缩，有损 |
| Transformer + KV cache | 随 $T$ 线性增长 | 保留在 token 表示里，无损（受 context 限制） |

这是一个非常干净的取舍：**压缩派省显存但丢信息，保留派不丢信息但要显存。**

### 4.2 一个容易被忽略的轴：时间维能不能并行

除了"保留 vs 压缩"，还有一个工程上极其重要的差别：

| 方法 | 时间维并行 | 原因 |
|---|---|---|
| Frame Stacking | 可以 | 窗口内所有帧一次 concat |
| RNN | 不行 | $h_t$ 依赖 $h_{t-1}$，必须逐步算 |
| LSTM | 不行 | 同上；gate 也依赖 $h_{t-1}$ |
| Transformer | 可以 | 所有位置一次矩阵乘算完 |

这是 RNN/LSTM 在 2017 年之后被 Transformer 取代的直接原因之一——**不是表达能力不够，而是无法利用 GPU 的并行度**。

代价被换到了另一头：Transformer 换来的是约 $O(T^2)$ 的计算量和 $O(T)$ 的 KV cache 显存。**并行不是免费的，它换成了对序列长度的平方依赖。**

## 5. History / Memory Mechanism 统一对比

| 方法 | 输入形式 | History 存在哪里 | 核心机制 | 长期依赖能力 | 主要问题 |
|---|---|---|---|---|---|
| Frame Stacking | $[B,T,D] \to [B,T\times D]$ | 原始 observation window | 直接拼接最近 $T$ 帧 | 弱，只能看固定窗口 | 超出窗口直接丢失；输入维度随 $T$ 增长 |
| RNN | $[B,T,D]$ | hidden state $h_t$ | 递归压缩历史 | 中等，理论可长期 | vanishing / exploding gradient；长期信息难训练 |
| LSTM | $[B,T,D]$ | $h_t + c_t$ | gate 控制保留、写入、输出 | 比 RNN 强 | 仍然把历史压缩进固定大小 state；顺序计算 |
| Transformer | $[B,T,D] \to [B,T,H]$ | historical token representations / KV cache | Attention 检索历史 | 强，长距离依赖路径短 | context window 有限；标准 attention 约 $O(T^2)$ |

读这张表时注意 $T$ 的两种含义：Frame Stacking 一行里 $T$ 指**窗口长度**（是超参数），其它行里指**序列长度**（是输入规模）。

核心演化关系可以记成：

$$
\boxed{ \text{Fixed Window} \rightarrow \text{Compressed Memory} \rightarrow \text{Gated Memory} \rightarrow \text{Attention Retrieval} }
$$


## 6. 怎么选：把 3.5 的判断顺序接到架构上

3.5 §6 给出的诊断顺序是：

```text
信息够不够 → 表示够不够 → 容量/优化够不够
```

本课回答的是其中第二步"表示"的具体选项。两层合起来：

| 问题 | 去哪一课找答案 | 判断依据 |
|---|---|---|
| 当前 observation 是否足以决定 action？ | 3.5 | 是否存在两个隐藏状态给出相同 $o_t$、却需要不同 action |
| 如果不够，history 用什么表示？ | 本课 | 保留还是压缩；时间维能否并行；成本随长度怎么增长 |
| 还不够，是不是容量/优化问题？ | 3.2 / 3.3 | train / validation loss 是否还有下降空间 |

对 policy 来说还有一个本课特有的细节：**history 不只包括 observation。** 3.5 §6 写的 memory 更新是

$$
m_t = \text{update}(m_{t-1},\ o_t,\ a_{t-1})
$$

它同时依赖上一时刻的 **action**。对机器人而言这很自然——"我上一帧做了什么"本身就是隐藏状态的一部分（例如夹爪是否已经闭合、是否已经抓到了物体）。所以决定把 history 拼进 policy 输入时，真正要定的是：

$$
o_{t-k:t} \quad \text{还是} \quad (o_{t-k:t},\ a_{t-k:t-1})
$$

一个诚实的边界（3.5 §7 已写过，这里只重申）：history / memory 解决的是"**信息不在当前 observation 里**"，**不解决数据不足或分布偏移**。Lesson 2 的失败属于后者（验证 MSE `0.2350` 劣于 mean-action baseline `0.1421`、held-out state 上 $|z|=13.21$、闭环 `0/10`），把窗口加长并不会自动修好它。

## 小结

四种方法的演化关系（§5 末尾那条链）在两条主线上展开：

| 主线 | Frame Stacking | RNN | LSTM | Transformer |
|---|---|---|---|---|
| history 是**保留**还是**压缩** | 保留（原始输入） | 压缩 | 压缩（但更可训） | 保留（token / KV） |
| history **存在哪里** | 输入 buffer | $h_t$ | $h_t, c_t$ | 每个 token 的表示 / KV cache |
| 信息损失 | 窗口外全丢 | 有瓶颈 | 有瓶颈 | 不丢（受 context 限制） |
| 时间维并行 | 可以 | 不行 | 不行 | 可以 |
| 成本随长度 | 输入维度线性增长 | 逐步线性 | 逐步线性 | 约 $O(T^2)$ + KV cache 显存 |

三条要带走的结论：

1. **架构不是重点，"信息在哪里"才是重点。** 两张表的所有差别都可以还原成一个问题：决定 action 所需的信息，此刻物理上存在于哪里？
2. **保留和压缩各有代价。** 压缩省显存省计算，但丢信息；保留不丢信息，但换来 $O(T^2)$ 的计算和对显存的线性需求。
3. **并行性是 RNN/LSTM 被 Transformer 取代的直接原因**，而 Transformer 为此付出的是对序列长度的平方依赖。

本课不含实验。它的实测部分在 3.5 Part B：在合成数据集上，单帧 MLP 的验证 MSE 卡在 `1.0027`（$=\operatorname{Var}(a\mid o)$，即信息上限），而 history MLP 降到 `1.3e-05`。本课是把那个结论拆到机制层面——3.5 说明**需要 history**，本课说明 **history 存在哪里、代价是什么**。

## 自检

1. RNN 和 LSTM 都属于"压缩派"。既然 LSTM 也是在压缩，为什么它能记住更长的历史？请给出一个涉及**梯度**的回答，而不是"因为有 gate"。
2. $QK^\top$ 得到的 $[T,T]$ 矩阵，和 $\operatorname{softmax}(QK^\top/\sqrt{d_k})$ 得到的 $[T,T]$ 矩阵，哪一个是"第 $i$ 个 token 对各 token 的关注程度"？另一个缺了哪条性质？
3. KV cache 让推理变快，但它把成本转移到了哪里？在这件事上 RNN 反而更"省"，那为什么不用 RNN？
4. 把 history 拼进 policy 输入时，只拼 $o_{t-k:t}$ 和同时拼 $(o_{t-k:t},\ a_{t-k:t-1})$，在什么任务上会真的不一样？请举一个机器人上的例子。
5. 某任务的单帧 policy 表现不好时，你要按什么顺序判断问题出在"信息"、"表示"还是"容量"？